# Forecast Ensembling | NeuralForecast
> Reduce forecast volatility by combining overlapping forecasts of the same target date.

## Motivation

A multi-horizon forecaster with step size smaller than the horizon produces overlapping forecasts: each target date is forecast from several forecast creation dates (FCDs). These overlapping forecasts can be ensembled to reduce volatility, i.e. how much the forecast of a date changes between FCDs (sEV, sFPC; see the [Forecast Volatility](../tutorials/forecast_volatility.html) tutorial).

There is a limit to this. Forecast uncertainty grows with lead time, so older forecasts of a date are less accurate and more variable than newer ones. Weighting all forecasts equally (`mean`) mixes in that uncertainty; weighting newer forecasts more (`ewm`) trades less volatility reduction for better accuracy.

Part 1 uses NumPy arrays from a synthetic forecaster; Part 2 uses NeuralForecast `cross_validation` output.

## The method

`ensemble_forecast_windows` replaces each forecast with a weighted combination of the forecasts of the same target date issued at the same or earlier FCDs. It is **causal**: later FCDs are never used.

Indexing by series $b$, FCD $t$ and horizon step $h$, with consecutive FCDs `stride` steps apart, the forecast of the same date issued $k$ FCDs earlier is $\hat{y}_{b,t-k,h+k \cdot \mathrm{stride}}$, and the ensembled forecast is

$$\tilde{y}_{b,t,h} = \frac{\sum_{k=0}^{K} w_k \, \hat{y}_{b,t-k,h+k \cdot \mathrm{stride}}}{\sum_{k=0}^{K} w_k}$$

where $K$ is the number of earlier forecasts available. Three methods choose the weights:

- **`mean`** weights every available forecast equally, $w_k = 1$.
- **`median`** takes the median of the same pool, which is robust to a single outlying forecast.
- **`ewm`** uses the exponential smoothing weights $w_k = (1-\alpha)^k$, as in `pandas.DataFrame.ewm(alpha=...)`, so the newest forecast weighs most. `alpha=1` keeps only the newest forecast, and as `alpha` approaches 0 it approaches `mean`.

`window_size` limits the pool to the most recent FCDs; `identity` returns the input unchanged.

## Setup

In [ ]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from neuralforecast import NeuralForecast
from neuralforecast.losses.numpy import (
    cross_validation_to_windows,
    excess_volatility,
    forecast_percentage_change,
    mae,
    mqloss,
)
from neuralforecast.losses.pytorch import MQLoss
from neuralforecast.models import MLP, NHITS
from neuralforecast.utils import AirPassengersPanel, ensemble_forecast_windows

warnings.filterwarnings("ignore")
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["font.family"] = "serif"

Forecasts are scored on accuracy (MAE on the median, sCRPS on the quantiles) and volatility (sEV, sFPC). Lower is better for all four.

In [ ]:
def evaluate(y, y_hat, quantiles, stride, mask=None):
    """Accuracy and volatility of a [B, T, H, C, Q] quantile forecast."""
    weights = np.ones_like(y) if mask is None else mask
    median = y_hat[..., quantiles.index(0.5)]
    scrps = 2 * mqloss(y, y_hat, quantiles=np.array(quantiles), weights=weights)
    scrps /= np.average(np.abs(y), weights=weights)
    return {
        "MAE": mae(y, median, weights=weights),
        "sCRPS": scrps,
        "sEV": excess_volatility(y=y, y_hat=y_hat, quantiles=quantiles, stride=stride, mask=mask),
        "sFPC": forecast_percentage_change(y_hat=median, stride=stride, mask=mask),
    }


METHODS = {
    "identity": {"method": "identity"},
    "mean": {"method": "mean"},
    "median": {"method": "median"},
    "ewm (alpha=0.8)": {"method": "ewm", "alpha": 0.8},
    "ewm (alpha=0.5)": {"method": "ewm", "alpha": 0.5},
    "ewm (alpha=0.3)": {"method": "ewm", "alpha": 0.3},
}

## 1. Ensembling on NumPy arrays

### A forecaster that revises on noise

One week of hourly demand, forecast every hour with a 48-hour horizon. The forecaster is unbiased, with independent noise at each FCD that grows with lead time.

In [ ]:
HORIZON = 48
STEP_SIZE = 1
N_HOURS = 168
FCDS = list(range(0, N_HOURS - HORIZON + 1, STEP_SIZE))

QUANTILES = [0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9]
Z = {0.1: -1.28, 0.25: -0.67, 0.4: -0.25, 0.5: 0.0, 0.6: 0.25, 0.75: 0.67, 0.9: 1.28}

hours = np.arange(N_HOURS)


def daily_shape(h):
    t = (h % 24) / 24
    return (
        40
        + 18 * np.sin(np.pi * (t - 0.05))
        + 10 * np.exp(-((t - 0.83) ** 2) / 0.01)
        - 6 * np.exp(-((t - 0.15) ** 2) / 0.008)
    )


weekday_effect = np.array([1.0, 1.02, 1.03, 1.02, 1.04, 0.92, 0.88])[hours // 24]
truth = daily_shape(hours) * weekday_effect + np.random.default_rng(7).normal(0, 0.5, N_HOURS)

# fresh error at every FCD, with a spread that grows with the lead time
rng = np.random.default_rng(77)
error_sd = 2.0 + 0.05 * np.arange(HORIZON)
medians = np.stack([truth[s : s + HORIZON] + rng.normal(0, error_sd) for s in FCDS])

y = np.stack([truth[s : s + HORIZON] for s in FCDS])[None, :, :, None]  # [B, T, H, C]
y_hat = np.stack([medians + Z[q] * error_sd for q in QUANTILES], axis=-1)[None, :, :, None]  # [B, T, H, C, Q]
print("y:    ", y.shape, "  (series, windows, horizon, targets)")
print("y_hat:", y_hat.shape)

The output has the same shape as the input:

In [ ]:
ensembled = ensemble_forecast_windows(y_hat, stride=STEP_SIZE, method="mean")
print("ensembled:", ensembled.shape)

Six consecutive FCDs before and after ensembling (median and 80% interval):

In [ ]:
lo, med, hi = (QUANTILES.index(q) for q in (0.1, 0.5, 0.9))
pick = list(range(72, 78))  # six consecutive FCDs, 1 hour apart
colors = plt.cm.viridis(np.linspace(0, 0.9, len(pick)))

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, sharey=True)
for ax, panel, title in [
    (axes[0], y_hat, "Six consecutive FCDs"),
    (axes[1], ensembled, "The same FCDs, ensembled (mean)"),
]:
    ax.plot(hours, truth, color="black", linewidth=1.2, label="actual", zorder=5)
    for i, color in zip(pick, colors):
        window = np.arange(FCDS[i], FCDS[i] + HORIZON)
        ax.plot(window, panel[0, i, :, 0, med], color=color, linewidth=1.4, label=f"FCD at hour {FCDS[i]}")
        ax.fill_between(window, panel[0, i, :, 0, lo], panel[0, i, :, 0, hi], color=color, alpha=0.2, linewidth=0)
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_ylabel("demand")

axes[0].legend(loc="lower left", fontsize=8, ncol=4, frameon=True)
axes[1].set_xlabel("hour")
plt.tight_layout()
plt.show()

Scores for each method:

In [ ]:
results = {
    name: evaluate(y, ensemble_forecast_windows(y_hat, stride=STEP_SIZE, **kwargs), QUANTILES, STEP_SIZE)
    for name, kwargs in METHODS.items()
}
pd.DataFrame(results).T.round(4)

All methods reduce volatility, and accuracy improves too, because this forecaster's errors are independent across FCDs and averaging cancels them. Real models' older forecasts are also staler, so accuracy can suffer.

## 2. Ensembling NeuralForecast models

`cross_validation_to_windows` converts `cross_validation` output into the arrays used above, along with the quantiles, mask and stride. Ensembling needs overlapping windows, so `step_size` must be smaller than `h`.

In [ ]:
df = AirPassengersPanel[["unique_id", "ds", "y"]]
HORIZON_AP = 12

models = [
    NHITS(h=HORIZON_AP, input_size=24, max_steps=100, loss=MQLoss(level=[80]),
          enable_progress_bar=False, logger=False, random_seed=0),
    MLP(h=HORIZON_AP, input_size=24, max_steps=100, loss=MQLoss(level=[80]),
        enable_progress_bar=False, logger=False, random_seed=0),
]

nf = NeuralForecast(models=models, freq="ME")
cv_df = nf.cross_validation(df=df, n_windows=16, step_size=2)
cv_df.head()

Ensemble and score each model, passing the mask:

In [ ]:
tables = {}
for model in ("NHITS", "MLP"):
    w = cross_validation_to_windows(cv_df, model=model)
    tables[model] = pd.DataFrame(
        {
            name: evaluate(
                w.y,
                ensemble_forecast_windows(w.y_hat, stride=w.stride, mask=w.mask, **kwargs),
                w.quantiles,
                w.stride,
                w.mask,
            )
            for name, kwargs in METHODS.items()
        }
    ).T

pd.concat(tables).round(4)

### Accuracy vs. stability

Volatility drops for both models; accuracy changes little and in either direction. Sweeping `alpha` shows the trade-off, relative to `identity`:

In [ ]:
alphas = [1.0, 0.8, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0.05]

fig, ax = plt.subplots(figsize=(8, 5))
for model, color in [("NHITS", "tab:blue"), ("MLP", "tab:orange")]:
    w = cross_validation_to_windows(cv_df, model=model)
    base = evaluate(w.y, w.y_hat, w.quantiles, w.stride, w.mask)

    def change(kwargs):
        ens = ensemble_forecast_windows(w.y_hat, stride=w.stride, mask=w.mask, **kwargs)
        scores = evaluate(w.y, ens, w.quantiles, w.stride, w.mask)
        return [100 * (scores[m] / base[m] - 1) for m in ("sEV", "MAE")]

    sweep = np.array([change({"method": "ewm", "alpha": a}) for a in alphas])
    ax.plot(sweep[:, 0], sweep[:, 1], "-o", color=color, markersize=4, label=f"{model}, ewm")
    for a, (x, yv) in zip(alphas, sweep):
        if model == "NHITS" and a in (1.0, 0.5, 0.1):
            ax.annotate(f"alpha={a}", (x, yv), textcoords="offset points", xytext=(5, 5), fontsize=8)
    for method, marker in [("mean", "s"), ("median", "D")]:
        x, yv = change({"method": method})
        ax.scatter(x, yv, marker=marker, color=color, s=50, label=f"{model}, {method}")

ax.axhline(0, color="grey", linewidth=0.8)
ax.set_xlabel("% change in sEV relative to identity (< 0 = less volatile)")
ax.set_ylabel("% change in MAE relative to identity (< 0 = more accurate)")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

Lowering `alpha` from 1 (no ensembling) cuts volatility quickly with little change in MAE; the cost, if any, appears near `mean`. Results vary with model, data and seed, so evaluate on your own validation windows.

- Works on `[B, T, H, C]` point and `[B, T, H, C, Q]` quantile forecasts.
- `window_size` bounds how stale the oldest pooled forecast can be.
- Causal, so it can run in production exactly as evaluated here.

## References

- Willa Potosnak, Malcolm Wolff, Mengfei Cao, Ruijun Ma, Tatiana Konstantinova, Dmitry Efimov, Michael W. Mahoney, Boris Oreshkin, Kin G. Olivares. [Forking-Sequences: Statistically and Computationally Efficient Multi-Horizon Forecasting with Reduced Volatility](https://openreview.net/forum?id=dXdycy7WCX). Transactions on Machine Learning Research (2026).